# 11 · Build the organizer

**Step 1.** `10_simulate_data` left a campaign on disk: four metadata modules
describing most of it, and four techniques in bare folders that nobody
described at all. This notebook builds the one object that knows where all of
it is.

```
Organizer("../OrgDemo/CuAu/cuau.json")      beside the data it organises
 └── sample_id              "CuAu05"
      ├── stages            synthesis · characterization · testing · computation
      ├── measurements      uvvis, eds, xps, … (ingested) · tem, sem, dls, tomo (linked by hand)
      └── derived           filled in by 12
```

The organizer is **one JSON file you name**. Not a directory layout, not a
hidden folder — a document you can copy, diff, version and email. The data
stays where it is; the file only records where that is. Here it sits inside
the campaign folder and records every path **relative to itself**, so the
folder — data, metadata and organizer — moves as one piece.

Two ways in, for two different situations:

| | |
|---|---|
| **ingest** | metadata somebody already wrote — a `*_dict.py` or the dict itself. Bulk, structured, re-readable. |
| **link** | data nobody wrote down. One call per measurement, by hand, because that is the only thing that is true. |

There is deliberately **no auto-attach**. The four bare folders here happen to
be named `<Modality>Data/<SampleID>/`, so `attach_folders()` would find them —
but almost no real campaign is laid out that way, and linking by hand is the
route that still works when it is not.

In [1]:
import runpy
import shutil

from NanoOrganizer import Organizer
from NanoOrganizer.core.pathmap import display_path
from NanoOrganizer.demo import demo_root

ROOT = demo_root("CuAu")              # ../OrgDemo/CuAu beside the repository
META = ROOT / "MetaData"

if not (META / "Synthesis_dict.py").exists():
    raise SystemExit("Run 10_simulate_data first.")

# This notebook is the recipe, so it starts from nothing: an organizer or
# stored fits left by an earlier run would carry into this one.
(ROOT / "cuau.json").unlink(missing_ok=True)
shutil.rmtree(ROOT / "results", ignore_errors=True)

org = Organizer(ROOT / "cuau.json", name="Cu-Au CO2RR library")
org

<Organizer 'Cu-Au CO2RR library': 0 samples, all — ../../OrgDemo/CuAu/cuau.json>

## Ingest what the people wrote down

One module per stage. `ingest` takes the file — a relative path is relative to
the organizer's own folder — and names the stage from the dict's variable
name: `Testing_dict` becomes the `testing` stage. Any block of a record that
**names files** becomes a measurement; everything else is kept verbatim as a
parameter and flattens into a dotted, filterable column.

In [2]:
for stage in ("Synthesis", "Characterization", "Testing", "Computation"):
    org.ingest(f"MetaData/{stage}_dict.py")

print(org.summary())
org.table()[["synthesis.composition.nominal_x_Au", "synthesis.status",
             "testing.performance.FE_CO_pct", "testing.performance.j_CO_mA_cm2",
             "computation.descriptors.E_ads_CO_eV", "n_measurements"]]

Cu-Au CO2RR library: 9 samples, 118 measurements (118 readable here, 0 not mounted).
Modalities: uvvis, eds, xps, raman, ir, saxs1d, waxs1d, saxs2d, xas, xpcs_g2, ec, dos
Root: ../../OrgDemo/CuAu


,synthesis.composition.nominal_x_Au,synthesis.status,testing.performance.FE_CO_pct,testing.performance.j_CO_mA_cm2,computation.descriptors.E_ads_CO_eV,n_measurements
sample_id,,,,,,
CuAu01,0.00,done,2.3,0.77,-0.620,17
CuAu02,0.10,done,5.4,1.71,-0.594,13
CuAu03,0.25,done,17.9,5.02,-0.555,15
CuAu04,0.40,done,44.0,10.74,-0.516,15
CuAu05,0.55,done,70.1,14.58,-0.477,15
CuAu06,0.70,done,82.6,14.21,-0.438,13
CuAu07,0.85,done,86.6,11.77,-0.399,15
CuAu08,1.00,done,87.6,8.76,-0.360,15
CuAu09,0.50,error,NaN,NaN,NaN,0


Twelve techniques came in through the dicts — spectra, scattering,
electrochemistry, DFT. `CuAu09` has a recipe and nothing else, which is
correct: the run aborted. TEM, SEM, DLS and tomography are not here yet.

In [3]:
org.catalog(counts=True)

,uvvis,eds,xps,raman,ir,saxs1d,waxs1d,saxs2d,xas,xpcs_g2,ec,dos
sample_id,,,,,,,,,,,,
CuAu01,14,1,2,1,1,1,1,2,2,1,9,1
CuAu02,14,1,2,1,1,1,1,0,0,0,9,1
CuAu03,14,1,2,1,1,1,1,0,2,0,9,1
CuAu04,14,1,2,1,1,1,1,2,0,1,9,1
CuAu05,14,1,2,1,1,1,1,0,2,0,9,1
CuAu06,14,1,2,1,1,1,1,0,0,0,9,1
CuAu07,14,1,2,1,1,1,1,0,2,0,9,1
CuAu08,14,1,2,1,1,1,1,2,0,1,9,1
CuAu09,0,0,0,0,0,0,0,0,0,0,0,0


## Editing a dict and re-ingesting

The dict is live. Load it, fix a number, pop a mistake — then ingest again.
`replace=True` makes the dict the **whole truth** for its stage: parameters
are replaced rather than merged, so a popped key is really gone instead of
lingering as a merge artefact that looks like data.

In [4]:
Synthesis_dict = runpy.run_path(str(META / "Synthesis_dict.py"))["Synthesis_dict"]

# The operator's paper log says CuAu04 was held 75 min, not 60. Fix the dict,
# and make it the whole truth for its stage again.
Synthesis_dict["CuAu04"]["conditions"]["hold_time_min"] = 75.0
org.ingest(synthesis=Synthesis_dict, replace=True)

org.table()[["synthesis.conditions.hold_time_min", "synthesis.status"]].T

sample_id,CuAu01,CuAu02,CuAu03,CuAu04,CuAu05,CuAu06,CuAu07,CuAu08,CuAu09
synthesis.conditions.hold_time_min,60.0,60.0,60.0,75.0,60.0,60.0,60.0,60.0,60.0
synthesis.status,done,done,done,done,done,done,done,done,error


## Link what nobody wrote down

Four techniques, a folder per sample. `link` takes a sample id, a technique,
and where the data is — relative to the organizer's folder, like everything
here — and reads the third argument three ways:

| | |
|---|---|
| a **folder** | listed now, filtered by the technique's extensions — a snapshot you can audit |
| a **glob** | kept live, re-expanded at read time, so frames written later appear |
| a **path or list** | stored verbatim |

Anything else passed to `link` is kept as metadata on the measurement — the
tomogram's voxel size, here, which nothing else recorded.

In [5]:
BY_HAND = {"tem": "TEMData", "sem": "SEMData", "dls": "DLSData", "tomo": "TomoData"}

for sample in org.ids():
    for modality, folder in BY_HAND.items():
        if not (ROOT / folder / sample).is_dir():
            continue          # not measured, or the run failed: leave the gap visible
        extra = {"voxel_size_nm": 2.0} if modality == "tomo" else {}
        org.link(sample, modality, f"{folder}/{sample}", stage="characterization", **extra)

print(org.summary())

Cu-Au CO2RR library: 9 samples, 144 measurements (144 readable here, 0 not mounted).
Modalities: uvvis, eds, xps, raman, ir, saxs1d, waxs1d, saxs2d, xas, xpcs_g2, ec, dos, tem, sem, dls, tomo
Root: ../../OrgDemo/CuAu


In [6]:
# note.txt sits beside the micrographs and is not a micrograph: the technique's
# declared extensions left it out without being asked.
tem = org.measurement("CuAu05", modality="tem")
print(len(tem.paths), "files:", [p.split("/")[-1] for p in tem.paths])

tomo = org.measurement("CuAu01", modality="tomo")
print("tomogram:", tomo.paths[0].split("/")[-1], "|", tomo.meta)

3 files: ['01.tif', '02.tif', '03.tif']
tomogram: tomogram.npy | {'voxel_size_nm': 2.0, 'linked': True}


## What is in it, and what is not

The gaps are the point. A measurement matrix is always sparse, and seeing
which comparisons exist beats discovering halfway through an argument that it
rests on a sample nobody measured.

In [7]:
org.catalog(counts=True)

,uvvis,eds,xps,raman,ir,saxs1d,waxs1d,saxs2d,xas,xpcs_g2,ec,dos,tem,sem,dls,tomo
sample_id,,,,,,,,,,,,,,,,
CuAu01,14,1,2,1,1,1,1,2,2,1,9,1,3,3,1,1
CuAu02,14,1,2,1,1,1,1,0,0,0,9,1,3,3,1,0
CuAu03,14,1,2,1,1,1,1,0,2,0,9,1,3,3,1,0
CuAu04,14,1,2,1,1,1,1,2,0,1,9,1,3,3,1,0
CuAu05,14,1,2,1,1,1,1,0,2,0,9,1,3,3,1,0
CuAu06,14,1,2,1,1,1,1,0,0,0,9,1,3,3,1,1
CuAu07,14,1,2,1,1,1,1,0,2,0,9,1,3,3,1,0
CuAu08,14,1,2,1,1,1,1,2,0,1,9,1,3,3,1,0
CuAu09,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## Paths are recorded as given — and relative ones travel

A relative path is relative to the organizer's folder, so everything in this
one resolves from wherever the folder is copied to, with nothing to configure.

Data that lives somewhere else is linked by its absolute path, recorded
exactly as given — a rewritten store would only work on the machine that wrote
it. Portability is then an alias: each such link registers the mount its data
sits on, so another machine edits one line per mount rather than one per
record.

```python
org.link("CuAu05", "xrd", "/mnt/beamline/2026-3/CuAu05/")          # elsewhere
org.project.add_alias("/mnt/beamline", ["/Volumes/beamline"])    # on a laptop
```

In [8]:
m = org.measurement("CuAu05", modality="waxs1d")
print("recorded:", m.paths[0])                                   # relative to cuau.json
print("resolves:", display_path(m.resolve(org.resolver)[0]))     # from here
print("aliases needed:", org.project.config.path_aliases)

report = org.available()
print(f"\n{report['n_available']}/{report['n_measurements']} measurements readable here")

recorded: Scattering/CuAu05/waxs_1d.dat
resolves: ../../OrgDemo/CuAu/Scattering/CuAu05/waxs_1d.dat
aliases needed: []

144/144 measurements readable here


## Anything else worth filtering on

`set_params` records what was never in a dict — a decision taken afterwards,
a grouping you only thought of once the results were in.

In [9]:
for sample in ("CuAu04", "CuAu05", "CuAu06"):
    org.set_params(sample, stage="testing", stability_test="queued")

org.table()[["testing.stability_test"]].T

sample_id,CuAu01,CuAu02,CuAu03,CuAu04,CuAu05,CuAu06,CuAu07,CuAu08,CuAu09
testing.stability_test,NaN,NaN,NaN,queued,queued,queued,NaN,NaN,NaN


## Save

One file. Links, parameters, aliases and — after `12` — derived values and
stored fits, all in it. The data is untouched and stays where it was.

In [10]:
print("saved", display_path(org.save()))
print(f"{org.path.stat().st_size / 1024:.0f} kB for "
      f"{len(org.project)} samples and {org.available()['n_measurements']} measurements")

saved ../../OrgDemo/CuAu/cuau.json
197 kB for 9 samples and 144 measurements


In [11]:
# The other export: every link as a table, re-importable with org.link_table().
# A spreadsheet is a better editor than a form when forty rows need the same fix.
org.links_table().head(6)

,sample_id,modality,source,stage,role,aux,instrument,note,n_frames,path_length_mm,...,q_invA,viscosity_Pa_s,temperature_K,technique,atmosphere,scan_rate_mV_s,electrolyte,hold_time_min,projection,voxel_size_nm
0,CuAu01,uvvis,RawSpectra/uvvis_runs/uvvis_b01_*.npy,characterization,,"{""wavelength_file"": ""RawSpectra/uvvis_runs/axi...",DemoSpec HR4000,in-situ growth series; the last frame is the p...,14.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CuAu01,eds,Spectroscopy/CuAu01/eds.dat,characterization,,NaN,DemoSEM + 100 mm2 SDD,"bulk composition; compare with XPS, which sees...",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,CuAu01,xps,Spectroscopy/CuAu01/xps_Cu2p.dat,characterization,cu2p,NaN,DemoXPS,the satellite near 942 eV is Cu(II),NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,CuAu01,xps,Spectroscopy/CuAu01/xps_Au4f.dat,characterization,au4f,NaN,DemoXPS,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,CuAu01,raman,Spectroscopy/CuAu01/raman.dat,characterization,,NaN,DemoRaman,Cu2O phonons below 700 cm-1; carbon D and G fr...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,CuAu01,ir,Spectroscopy/CuAu01/ir_seira.dat,characterization,,NaN,"DemoFTIR, ATR-SEIRAS cell",the band near 2100 cm-1 is CO adsorbed atop a ...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


---

**Next: `12_use_organizer`** — reopen this file and do the work: look, load,
visualise all four kinds of data, fit, batch, and check fifteen techniques
against each other and against the answer key.